In [9]:
"""
Plant Disease Detection - MobileNetV2 + 5-Fold Cross Validation
"""

import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import random
from datetime import datetime

# ------------------------- CONFIG -------------------------
DATASET_DIR = r"C:\Users\natra\Downloads\dl project dataset\PlantVillage"  
BATCH_SIZE = 32
EPOCHS = 10
N_SPLITS = 5
MAX_IMAGES_PER_CLASS = 400 
OUTPUT_DIR = "mobilenetv2_kfold_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------- LOAD DATA -------------------------
print("📦 Loading dataset...")

X = []
y = []
classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])

for label in classes:
    folder_path = os.path.join(DATASET_DIR, label)
    files = os.listdir(folder_path)
    random.shuffle(files)
    files = files[:MAX_IMAGES_PER_CLASS]

    for file in files:
        img_path = os.path.join(folder_path, file)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, IMAGE_SIZE)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        X.append(img)
        y.append(label)

X = np.array(X, dtype=np.float32) / 255.0
y = np.array(y)

print(f"✅ Loaded {len(X)} images from {len(classes)} classes (~{MAX_IMAGES_PER_CLASS} per class max).")

le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(classes)
print(f"Number of classes: {num_classes}")

# ------------------------- MODEL FUNCTION -------------------------
def build_mobilenet_model(input_shape=(224,224,3), num_classes=2, dropout_rate=0.4):
    base = MobileNetV2(include_top=False, weights='imagenet', input_shape=input_shape)
    base.trainable = False  # freeze base for transfer learning

    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    return Model(inputs=base.input, outputs=outputs)

# ------------------------- DATA AUGMENTATION -------------------------
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='reflect'
)

# ------------------------- 5-FOLD CROSS VALIDATION -------------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_encoded), 1):
    print(f"\n================ Fold {fold}/{N_SPLITS} ================")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]

    # Build model
    model = build_mobilenet_model(input_shape=(*IMAGE_SIZE,3), num_classes=num_classes, dropout_rate=0.4)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # Callbacks
    fold_time = datetime.now().strftime('%Y%m%d_%H%M%S')
    model_path = os.path.join(OUTPUT_DIR, f"mobilenetv2_fold{fold}_{fold_time}.keras")
    callbacks = [
        ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    ]

    # Train
    history = model.fit(
        datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    # Evaluate
    val_pred = np.argmax(model.predict(X_val), axis=1)
    acc = accuracy_score(y_val, val_pred)
    fold_accuracies.append(acc)
    print(f"✅ Fold {fold} Accuracy: {acc:.4f}")
    print(classification_report(y_val, val_pred, target_names=le.classes_))

    # Save accuracy/loss plots
    plt.figure(figsize=(8,4))
    plt.plot(history.history['accuracy'], label='train_acc')
    plt.plot(history.history['val_accuracy'], label='val_acc')
    plt.title(f'Fold {fold} Accuracy')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(OUTPUT_DIR, f'fold{fold}_accuracy.png'))
    plt.close()

    plt.figure(figsize=(8,4))
    plt.plot(history.history['loss'], label='train_loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.title(f'Fold {fold} Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(OUTPUT_DIR, f'fold{fold}_loss.png'))
    plt.close()

# ------------------------- FINAL SUMMARY -------------------------
print("\n====================== FINAL RESULTS ======================")
print(f"Average Accuracy over {N_SPLITS} folds: {np.mean(fold_accuracies):.4f}")
print(f"Standard Deviation: {np.std(fold_accuracies):.4f}")

summary_path = os.path.join(OUTPUT_DIR, "kfold_summary.txt")
with open(summary_path, "w") as f:
    for i, acc in enumerate(fold_accuracies, 1):
        f.write(f"Fold {i}: {acc:.4f}\n")
    f.write(f"\nAverage Accuracy: {np.mean(fold_accuracies):.4f}\n")
    f.write(f"Std Dev: {np.std(fold_accuracies):.4f}\n")

print(f"\n📁 All results saved to: {OUTPUT_DIR}")


📦 Loading dataset...
✅ Loaded 4125 images from 11 classes (~400 per class max).
Number of classes: 11

================ Fold 1/5 ================
Epoch 1/10


C:\Users\natra\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 860ms/step - accuracy: 0.2766 - loss: 2.2703
Epoch 1: val_accuracy improved from -inf to 0.89818, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold1_20251014_185539.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - accuracy: 0.2781 - loss: 2.2652 - val_accuracy: 0.8982 - val_loss: 0.6145 - learning_rate: 1.0000e-04
Epoch 2/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 828ms/step - accuracy: 0.7107 - loss: 0.8742
Epoch 2: val_accuracy improved from 0.89818 to 0.94061, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold1_20251014_185539.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 103s 954ms/step - accuracy: 0.7111 - loss: 0.8731 - val_accuracy: 0.9406 - val_loss: 0.3030 - learning_rate: 1.0000e-04
Epoch 3/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 810ms/step - accuracy: 0.8255 - loss: 0.5447
Epoch 3: val_accuracy improved from 0.94061 to 0.94788, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold1_20251014_185539.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 101s 931ms/s

C:\Users\natra\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 777ms/step - accuracy: 0.2699 - loss: 2.2879
Epoch 1: val_accuracy improved from -inf to 0.85212, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold2_20251014_191351.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 112s 925ms/step - accuracy: 0.2715 - loss: 2.2824 - val_accuracy: 0.8521 - val_loss: 0.6433 - learning_rate: 1.0000e-04
Epoch 2/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 785ms/step - accuracy: 0.7431 - loss: 0.8195
Epoch 2: val_accuracy improved from 0.85212 to 0.92606, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold2_20251014_191351.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 142s 924ms/step - accuracy: 0.7433 - loss: 0.8185 - val_accuracy: 0.9261 - val_loss: 0.3196 - learning_rate: 1.0000e-04
Epoch 3/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 822ms/step - accuracy: 0.8002 - loss: 0.5704
Epoch 3: val_accuracy improved from 0.92606 to 0.94182, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold2_20251014_191351.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 103s 955m

...

104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 799ms/step - accuracy: 0.2647 - loss: 2.3469
Epoch 1: val_accuracy improved from -inf to 0.87636, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold3_20251014_193153.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 112s 947ms/step - accuracy: 0.2662 - loss: 2.3413 - val_accuracy: 0.8764 - val_loss: 0.6357 - learning_rate: 1.0000e-04
Epoch 2/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 784ms/step - accuracy: 0.7184 - loss: 0.8494
Epoch 2: val_accuracy improved from 0.87636 to 0.93455, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold3_20251014_193153.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 99s 905ms/step - accuracy: 0.7187 - loss: 0.8484 - val_accuracy: 0.9345 - val_loss: 0.3336 - learning_rate: 1.0000e-04
Epoch 3/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 752ms/step - accuracy: 0.8380 - loss: 0.5301
Epoch 3: val_accuracy improved from 0.93455 to 0.94545, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold3_20251014_193153.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 96s 882ms/

C:\Users\natra\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 778ms/step - accuracy: 0.2877 - loss: 2.2576
Epoch 1: val_accuracy improved from -inf to 0.90061, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold4_20251014_194942.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 110s 919ms/step - accuracy: 0.2892 - loss: 2.2522 - val_accuracy: 0.9006 - val_loss: 0.5532 - learning_rate: 1.0000e-04
Epoch 2/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 764ms/step - accuracy: 0.7158 - loss: 0.8413
Epoch 2: val_accuracy improved from 0.90061 to 0.94667, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold4_20251014_194942.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 138s 884ms/step - accuracy: 0.7161 - loss: 0.8405 - val_accuracy: 0.9467 - val_loss: 0.2783 - learning_rate: 1.0000e-04
Epoch 3/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 786ms/step - accuracy: 0.8232 - loss: 0.5286
Epoch 3: val_accuracy improved from 0.94667 to 0.95273, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold4_20251014_194942.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 98s 898ms

C:\Users\natra\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step - accuracy: 0.2788 - loss: 2.2446
Epoch 1: val_accuracy improved from -inf to 0.87515, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold5_20251014_200104.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 47s 404ms/step - accuracy: 0.2802 - loss: 2.2395 - val_accuracy: 0.8752 - val_loss: 0.6546 - learning_rate: 1.0000e-04
Epoch 2/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 341ms/step - accuracy: 0.7153 - loss: 0.8435
Epoch 2: val_accuracy improved from 0.87515 to 0.92606, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold5_20251014_200104.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 43s 394ms/step - accuracy: 0.7156 - loss: 0.8427 - val_accuracy: 0.9261 - val_loss: 0.3388 - learning_rate: 1.0000e-04
Epoch 3/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 337ms/step - accuracy: 0.8322 - loss: 0.5558
Epoch 3: val_accuracy improved from 0.92606 to 0.94667, saving model to mobilenetv2_kfold_outputs\mobilenetv2_fold5_20251014_200104.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 42s 393ms/s